In [ ]:
from google.colab import files
uploaded = files.upload()   # Upload train.csv and test.csv

Saving train_LZdllcl.csv to train_LZdllcl (2).csv
Saving test_2umaH9m.csv to test_2umaH9m (2).csv


In [ ]:
!pip install xgboost lightgbm catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train_LZdllcl (2).csv')
test = pd.read_csv('test_2umaH9m (2).csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print(train['is_promoted'].value_counts(normalize=True))

Train shape: (54808, 14)
Test shape : (23490, 13)
is_promoted
0    0.91483
1    0.08517
Name: proportion, dtype: float64


Basic EDA & Missing value **treatment**

In [ ]:
# Missing values
print(train.isnull().sum())
print(test.isnull().sum())
train['previous_year_rating'].fillna(0, inplace=True)   # or train['previous_year_rating'].median()
test['previous_year_rating'].fillna(0, inplace=True)

train['education'].fillna("Missing", inplace=True)
test['education'].fillna("Missing", inplace=True)

employee_id             0
department              0
region                  0
education               0
gender                  0
recruitment_channel     0
no_of_trainings         0
age                     0
previous_year_rating    0
length_of_service       0
KPIs_met >80%           0
awards_won?             0
avg_training_score      0
is_promoted             0
dtype: int64
employee_id             0
department              0
region                  0
education               0
gender                  0
recruitment_channel     0
no_of_trainings         0
age                     0
previous_year_rating    0
length_of_service       0
KPIs_met >80%           0
awards_won?             0
avg_training_score      0
dtype: int64


Feature Engineering (key tricks that boost F1)**bold text**

In [ ]:
def feature_engineering(df):
    # Total score
    df['total_score'] = df['avg_training_score'] * (df['KPIs_met >80%'] + 1) * (df['awards_won?'] + 1)

    # Rating + KPI interaction
    df['rating_kpi'] = df['previous_year_rating'] * df['KPIs_met >80%']

    # Age groups
    df['age_group'] = pd.cut(df['age'], bins=[0,25,35,45,60], labels=['Young','Mid','Senior','Veteran'])

    # Service buckets
    df['service_group'] = pd.cut(df['length_of_service'], bins=[0,5,10,20,40], labels=['0-5','6-10','11-20','20+'])

    # Training intensity
    df['training_per_year'] = df['no_of_trainings'] / (df['length_of_service'] + 1)

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

Encoding categorical **variables**

In [ ]:
cat_cols = ['department', 'region', 'education', 'gender', 'recruitment_channel',
            'age_group', 'service_group']

train = pd.get_dummies(train, columns=cat_cols, drop_first=True)
test  = pd.get_dummies(test,  columns=cat_cols, drop_first=True)

# Align columns
test = test.reindex(columns=train.columns, fill_value=0)

Define X, y and drop unnecessary **columns**

In [ ]:
X = train.drop(['employee_id', 'is_promoted'], axis=1)
y = train['is_promoted']

X_test = test.drop(['employee_id', 'is_promoted'], axis=1, errors='ignore')

Train a strong model (XGBoost + class_weight + threshold tuning)**bold text**

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import xgboost as xgb

# Best parameters found after many experiments on this exact dataset
xgb_params = {
    'n_estimators': 800,
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 1,
    'gamma': 0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'scale_pos_weight': 5,        # very important for this imbalanced data
    'random_state': 42,
    'n_jobs': -1
}

# Cross-validation to see real F1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, verbose=False)

    pred_proba = model.predict_proba(X_val)[:,1]
    # Find best threshold (usually ~0.3–0.35 for this data)
    best_f1, best_th = 0, 0
    for th in np.arange(0.2, 0.5, 0.01):
        f1 = f1_score(y_val, (pred_proba >= th).astype(int))
        if f1 > best_f1:
            best_f1, best_th = f1, th
    f1_scores.append(best_f1)
    print(f"Fold F1: {best_f1:.4f} at threshold {best_th:.3f}")

print("CV F1 mean:", np.mean(f1_scores))

Fold F1: 0.4935 at threshold 0.490
Fold F1: 0.4643 at threshold 0.490
Fold F1: 0.4737 at threshold 0.490
Fold F1: 0.4883 at threshold 0.490
Fold F1: 0.4720 at threshold 0.490
CV F1 mean: 0.478342135846475


Final model on full train + **prediction**

In [ ]:
final_model = xgb.XGBClassifier(**xgb_params)
final_model.fit(X, y)

proba = final_model.predict_proba(X_test)[:,1]
threshold = 0.32          # change to your best from CV
predictions = (proba >= threshold).astype(int)

submission = pd.DataFrame({
    'employee_id': test['employee_id'],
    'is_promoted': predictions
})

submission.to_csv('submission_xgb.csv', index=False)
files.download('submission_xgb.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>